<h1 style="font-size: 36px">1. Setup and Data Loading </h1>

In [ ]:
#Necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier 
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif
import joblib

#training data Load
train_data = pd.read_csv(r'C:\Users\ADMIN\OneDrive\Desktop\Customer Churn Prediction\Data\Processed\cleaned_customer_churn_data.csv')

#Preprocessor Load
preprocessor = joblib.load('C:\\Users\\ADMIN\\OneDrive\\Desktop\\Customer Churn Prediction\\Models\\preprocessor.pkl')

<h1 style="font-size: 36px">2.Data Set Split </h1>

In [ ]:
X_train = train_data.drop('Churn', axis=1)  # Separating features
y_train = train_data['Churn'].map({'Yes': 1, 'No': 0})  # Encoding target variable
#Train test split
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

<h1 style="font-size: 36px">3.Model Training </h1>

In [11]:
#Creating a pipeline that includes the preprocessor and the Random Forest Classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('Feature_Selection', SelectKBest(f_classif, k=15)),  # clean & simple
    ('classifier', RandomForestClassifier(
        class_weight='balanced',  # handles churn imbalance
        random_state=42
    ))
])

In [12]:
#Hyperparameter tuning for Random Forest model
param_grid = {
    'Feature_Selection__k': [10, 15, 20],      
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, 15],      
    'classifier__max_samples': [0.6, 0.8],
    'classifier__min_samples_split': [5, 10],
    'classifier__min_samples_leaf': [4, 8],       
}

grid_search_rf = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

In [13]:
#Training the model
grid_search_rf.fit(X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'Feature_Selection__k': [10, 15, ...], 'classifier__max_depth': [5, 10, ...], 'classifier__max_samples': [0.6, 0.8], 'classifier__min_samples_leaf': [4, 8], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the 

In [15]:
#Evaluating the model
y_pred_rf = grid_search_rf.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Accuracy Score:", accuracy_score(y_test, y_pred_rf))
print("ROC AUC Score:", roc_auc_score(y_test, grid_search_rf.predict_proba(X_test)[:, 1]))
    

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.75      0.81      1033
           1       0.52      0.74      0.61       374

    accuracy                           0.75      1407
   macro avg       0.70      0.74      0.71      1407
weighted avg       0.79      0.75      0.76      1407

Confusion Matrix:
 [[778 255]
 [ 99 275]]
Accuracy Score: 0.7484008528784648
ROC AUC Score: 0.8320865968494235


In [16]:
print(f"Random Forest Best Params: {grid_search_rf.best_params_}")

Random Forest Best Params: {'Feature_Selection__k': 20, 'classifier__max_depth': 10, 'classifier__max_samples': 0.6, 'classifier__min_samples_leaf': 8, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}


In [25]:
#Creatring a pipeline that includes the preprocessor and the CatBoost Classifier
pipeline_cb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', CatBoostClassifier(
        verbose=0,
        eval_metric='AUC',
        auto_class_weights='Balanced',
    ))
])
#Hyperparameter tuning for CatBoost model
param_grid_cb = {
    'classifier__iterations': [300, 500, 800],
    'classifier__depth': [6, 8, 10],
    'classifier__learning_rate': [0.03, 0.05, 0.1],
    'classifier__l2_leaf_reg': [3, 5, 7],
    'classifier__border_count': [64, 128]
}
grid_search_cb = GridSearchCV(estimator=pipeline_cb, param_grid=param_grid_cb, cv=5, n_jobs=-1, scoring='roc_auc', verbose=1)
#Training the CatBoost model
grid_search_cb.fit(X_train, y_train)
#Evaluating the CatBoost model
y_pred_cb = grid_search_cb.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_cb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_cb))
print("Accuracy Score:", accuracy_score(y_test, y_pred_cb))
print("ROC AUC Score:", roc_auc_score(y_test, grid_search_cb.predict_proba(X_test)[:, 1]))


Fitting 5 folds for each of 162 candidates, totalling 810 fits
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.73      0.81      1033
           1       0.51      0.78      0.62       374

    accuracy                           0.74      1407
   macro avg       0.71      0.76      0.71      1407
weighted avg       0.80      0.74      0.76      1407

Confusion Matrix:
 [[754 279]
 [ 82 292]]
Accuracy Score: 0.7434257285003554
ROC AUC Score: 0.8330844174332586


In [26]:
print(f"CatBoost Best Params: {grid_search_cb.best_params_}")

CatBoost Best Params: {'classifier__border_count': 64, 'classifier__depth': 6, 'classifier__iterations': 300, 'classifier__l2_leaf_reg': 7, 'classifier__learning_rate': 0.03}


In [5]:
#Creating a pipeline that includes XGBoost Classifier and Preprocessor
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        random_state=42,
        eval_metric='auc',
        verbosity=0
    ))
])

xgb_param_grid = {
    'classifier__n_estimators':   [100, 200],
    'classifier__max_depth':      [3, 5, 7],
    'classifier__learning_rate':  [0.01, 0.05, 0.1],
    'classifier__subsample':      [0.7, 0.8],
    'classifier__colsample_bytree': [0.7, 0.8],
    'classifier__scale_pos_weight': [sum(y_train==0)/sum(y_train==1)]  
}

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

xgb_train_auc = roc_auc_score(y_train, xgb_grid.predict_proba(X_train)[:,1])
xgb_test_auc  = roc_auc_score(y_test,  xgb_grid.predict_proba(X_test)[:,1])
print(f"XGBoost Best Params: {xgb_grid.best_params_}")
print(f"XGBoost  → Train AUC: {xgb_train_auc:.3f} | Test AUC: {xgb_test_auc:.3f} | Gap: {xgb_train_auc - xgb_test_auc:.3f}")


Fitting 5 folds for each of 72 candidates, totalling 360 fits
XGBoost Best Params: {'classifier__colsample_bytree': 0.7, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 100, 'classifier__scale_pos_weight': 2.762541806020067, 'classifier__subsample': 0.7}
XGBoost  → Train AUC: 0.867 | Test AUC: 0.835 | Gap: 0.032


In [19]:
#Creating a pipeline that includes LightBGM Classifier and Preprocessor
lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(
        random_state=42,
        verbose=-1
    ))
])

lgbm_param_grid = {
    'classifier__n_estimators':  [100, 200],
    'classifier__max_depth':     [4, 6, 8],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample':     [0.7, 0.8],
    'classifier__colsample_bytree': [0.7, 0.8],
    'classifier__class_weight':  ['balanced']  # fixed, not tuned
}

lgbm_grid = GridSearchCV(
    estimator=lgbm_pipeline,
    param_grid=lgbm_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

lgbm_grid.fit(X_train, y_train)

lgbm_train_auc = roc_auc_score(y_train, lgbm_grid.predict_proba(X_train)[:,1])
lgbm_test_auc  = roc_auc_score(y_test,  lgbm_grid.predict_proba(X_test)[:,1])
print(f"LightGBM Best Params: {lgbm_grid.best_params_}")
print(f"LightGBM → Train AUC: {lgbm_train_auc:.3f} | Test AUC: {lgbm_test_auc:.3f} | Gap: {lgbm_train_auc - lgbm_test_auc:.3f}")

Fitting 5 folds for each of 72 candidates, totalling 360 fits
LightGBM Best Params: {'classifier__class_weight': 'balanced', 'classifier__colsample_bytree': 0.7, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 4, 'classifier__n_estimators': 100, 'classifier__subsample': 0.7}
LightGBM → Train AUC: 0.880 | Test AUC: 0.833 | Gap: 0.046


c:\Users\ADMIN\OneDrive\Desktop\Customer Churn Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\ADMIN\OneDrive\Desktop\Customer Churn Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [28]:
#Defining invidual with best parameters


xgb_best = XGBClassifier(
    colsample_bytree=0.7,
    learning_rate=0.05,
    max_depth=3,
    n_estimators=100,
    scale_pos_weight=2.762541806020067,
    subsample=0.7,
    random_state=42,
    eval_metric='auc',
    verbosity=0
)

lgbm_best = LGBMClassifier(
    class_weight='balanced',
    colsample_bytree=0.7,
    learning_rate=0.05,
    max_depth=4,
    n_estimators=100,
    subsample=0.7,
    random_state=42,
    verbose=-1
)

cat_best = CatBoostClassifier(
    border_count= 64, 
    depth= 6, 
    iterations= 300, 
    l2_leaf_reg= 7,
    learning_rate= 0.03
)





In [30]:
#Creating an Ensemble Voting Pipeline and Preprocessor
ensemble_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', VotingClassifier(
        estimators=[
            ('xgb',  xgb_best),
            ('lgbm', lgbm_best),
            ('cat',  cat_best),
        ],
        voting='soft',  # uses probabilities — better for AUC
        weights=[2, 1, 1]  # give XGBoost more weight as it's the best
    ))
])

In [31]:
#Train the ensemble voting pipeline
ensemble_pipeline.fit(X_train, y_train)

0:	learn: 0.6742610	total: 8.25ms	remaining: 2.47s
1:	learn: 0.6570728	total: 12.1ms	remaining: 1.8s
2:	learn: 0.6407179	total: 15.4ms	remaining: 1.52s
3:	learn: 0.6238543	total: 21.3ms	remaining: 1.58s
4:	learn: 0.6096385	total: 25.1ms	remaining: 1.48s
5:	learn: 0.5950706	total: 29ms	remaining: 1.42s
6:	learn: 0.5809872	total: 32.3ms	remaining: 1.35s
7:	learn: 0.5702761	total: 38.3ms	remaining: 1.4s
8:	learn: 0.5604746	total: 41.7ms	remaining: 1.35s
9:	learn: 0.5511399	total: 46.9ms	remaining: 1.36s
10:	learn: 0.5416518	total: 51.8ms	remaining: 1.36s
11:	learn: 0.5329952	total: 56.3ms	remaining: 1.35s
12:	learn: 0.5253551	total: 59.8ms	remaining: 1.32s
13:	learn: 0.5178272	total: 63.1ms	remaining: 1.29s
14:	learn: 0.5116407	total: 69.9ms	remaining: 1.33s
15:	learn: 0.5055514	total: 76.7ms	remaining: 1.36s
16:	learn: 0.4991618	total: 84.1ms	remaining: 1.4s
17:	learn: 0.4942541	total: 89.9ms	remaining: 1.41s
18:	learn: 0.4893507	total: 93.2ms	remaining: 1.38s
19:	learn: 0.4842625	total:

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('binary', ...), ('onehot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the di

In [32]:
# ── Evaluate ──────────────────────────────────────────────────
train_auc = roc_auc_score(y_train, ensemble_pipeline.predict_proba(X_train)[:,1])
test_auc  = roc_auc_score(y_test,  ensemble_pipeline.predict_proba(X_test)[:,1])

print(f"Ensemble → Train AUC: {train_auc:.3f} | Test AUC: {test_auc:.3f} | Gap: {train_auc - test_auc:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, ensemble_pipeline.predict(X_test)))

Ensemble → Train AUC: 0.879 | Test AUC: 0.835 | Gap: 0.045

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.75      0.81      1033
           1       0.52      0.74      0.61       374

    accuracy                           0.75      1407
   macro avg       0.70      0.74      0.71      1407
weighted avg       0.79      0.75      0.76      1407



c:\Users\ADMIN\OneDrive\Desktop\Customer Churn Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\ADMIN\OneDrive\Desktop\Customer Churn Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\ADMIN\OneDrive\Desktop\Customer Churn Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Since all the models are having similer accuracy and auc-roc score but the XGBoost model is having the least amount of difference between Training Accuracy and Testing Accuraccy ,The XGBoost pipeline is chosen to be the Telecom Churn Prediction Model.

<h1 style="font-size: 36px">4.Model Save </h1>

In [ ]:
#Saving the best model of XGBoost
joblib.dump(xgb_pipeline,'best_xgb_model.pkl')

['Telecom_Churn_Prediction_Model.pkl']